# Optimized Boston — Model Comparison

Trains three variants on the same RM/LSTAT/PTRATIO features and persists each:
- Baseline neural net (no callbacks)
- + Early stopping
- + Early stopping + extra hidden layer

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

## Data

In [ ]:
df = pd.read_csv('datasets/Boston1.csv')
df.rename(columns={'medv': 'price'}, inplace=True)
X = df[['rm', 'lstat', 'ptratio']]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## Helpers

In [ ]:
def evaluate(model, X, y):
    p = model.predict(X, verbose=0).flatten()
    return dict(
        mae=mean_absolute_error(y, p),
        rmse=float(np.sqrt(mean_squared_error(y, p))),
        r2=r2_score(y, p),
    )

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

## Variant 1 — baseline

In [ ]:
def make_model(extra_layer=False):
    layers = [Input(shape=(X_train_s.shape[1],)),
              Dense(32, activation='relu'),
              Dense(16, activation='relu')]
    if extra_layer:
        layers.append(Dense(8, activation='relu'))
    layers.append(Dense(1))
    m = Sequential(layers)
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

baseline = make_model()
baseline.fit(X_train_s, y_train, epochs=300, batch_size=16,
             validation_split=0.2, verbose=0)
m_baseline = evaluate(baseline, X_test_s, y_test)
m_baseline

## Variant 2 — early stopping

In [ ]:
es_model = make_model()
es_model.fit(X_train_s, y_train, epochs=300, batch_size=16,
             validation_split=0.2, verbose=0, callbacks=[early_stop])
m_es = evaluate(es_model, X_test_s, y_test)
m_es

## Variant 3 — early stopping + extra layer

In [ ]:
es_extra = make_model(extra_layer=True)
es_extra.fit(X_train_s, y_train, epochs=300, batch_size=16,
             validation_split=0.2, verbose=0, callbacks=[early_stop])
m_extra = evaluate(es_extra, X_test_s, y_test)
m_extra

## Comparison

In [ ]:
results = pd.DataFrame({
    'baseline':           m_baseline,
    'early_stop':         m_es,
    'early_stop_extra':   m_extra,
}).T
results

## Save best

In [ ]:
# Pick model with lowest MAE
best_name, best_model = min(
    [('baseline', baseline), ('early_stop', es_model), ('early_stop_extra', es_extra)],
    key=lambda kv: evaluate(kv[1], X_test_s, y_test)['mae'],
)
MODELS_DIR = Path('models'); MODELS_DIR.mkdir(parents=True, exist_ok=True)
best_model.save(MODELS_DIR / 'boston_optimized.keras')
joblib.dump(scaler, MODELS_DIR / 'boston_optimized_scaler.pkl')
print(f'Best variant: {best_name}')
print('Saved -> models/boston_optimized.keras')